In [1]:
"""
replay_ekf_asrun.py -- Completa el replay del EKF con las ganancias as-run
sobre los cinco CSVs EKF, para tener el número de respaldo si un revisor lo pide.
"""
import os, glob, re
import numpy as np
import pandas as pd

Ts, EPS = 2.0, 1e-10
P = {'Atank': 3.02e-3, 'rho': 997.0, 'eta': 8.9e-4, 'g': 9.81,
     'alphao1': 0.0271, 'Do1': 10e-3, 'alphao2': 0.0271, 'A2': 7.85e-5,
     'alphao3': 0.0271, 'Do3': 10e-3, 'alpha120': 0.3038, 'D12': 10e-3,
     'A12': 7.85e-5, 'lambdac12': 24000, 'alpha230': 0.1344, 'D23': 10e-3,
     'A23': 7.85e-5, 'lambdac23': 29600, 'qi1max': 2.69e-5, 'qi3max': 2.41e-5}
CSV_DIR = r"C:\Users\PC\Downloads\articulo"

def nl_ode(x, u, p=P):
    h1 = max(x[0], EPS); h2 = max(x[1], EPS); h3 = max(x[2], EPS)
    qi1 = p['qi1max']*np.clip(u[0],0,1); qi3 = p['qi3max']*np.clip(u[1],0,1)
    qo1 = p['alphao1']*(p['Do1']**2*np.pi/4)*np.sqrt(2*p['g']*h1)
    qo2 = p['alphao2']*p['A2']*np.sqrt(2*p['g']*h2)
    qo3 = p['alphao3']*(p['Do3']**2*np.pi/4)*np.sqrt(2*p['g']*h3)
    d12 = h1-h2; l12 = p['D12']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(d12)+EPS)
    q12 = p['alpha120']*np.tanh(2*l12/p['lambdac12'])*p['A12']*np.sqrt(2*p['g']*abs(d12)+EPS)*np.sign(d12+1e-15)
    d23 = h2-h3; l23 = p['D23']*p['rho']/p['eta']*np.sqrt(2*p['g']*abs(d23)+EPS)
    q23 = p['alpha230']*np.tanh(2*l23/p['lambdac23'])*p['A23']*np.sqrt(2*p['g']*abs(d23)+EPS)*np.sign(d23+1e-15)
    return np.array([(qi1-q12-qo1)/p['Atank'],(q12-q23-qo2)/p['Atank'],(qi3+q23-qo3)/p['Atank']])

def rk4(x, u, dt=Ts):
    k1=nl_ode(x,u); k2=nl_ode(x+dt/2*k1,u); k3=nl_ode(x+dt/2*k2,u); k4=nl_ode(x+dt*k3,u)
    return np.clip(x+dt/6*(k1+2*k2+2*k3+k4), 0, 0.27)

def outlier_filter(x, max_delta=0.05):
    y = np.array(x, float)
    for k in range(1, len(y)):
        if abs(y[k]-y[k-1]) > max_delta: y[k] = y[k-1]
    return y

def mask_paper(T):
    m = np.ones(len(T), dtype=bool)
    for tc in (0.0, 264.0, 528.0): m &= ~((T >= tc) & (T < tc+60.0))
    return m

class EKF:
    C = np.array([[1.,0,0],[0,0,1.]])
    def __init__(s, x0, P0, Q, R): s.x=np.array(x0,float); s.P=P0.copy(); s.Q=Q; s.R=R
    def predict(s, u, eps=1e-6):
        f0 = nl_ode(s.x,u); J = np.zeros((3,3))
        for i in range(3):
            xp = s.x.copy(); xp[i] += eps
            J[:,i] = (nl_ode(xp,u) - f0)/eps
        F = np.eye(3) + Ts*J
        s.x = rk4(s.x, u); s.P = F @ s.P @ F.T + s.Q
    def update(s, y):
        S = s.C @ s.P @ s.C.T + s.R; K = s.P @ s.C.T @ np.linalg.inv(S)
        s.x = np.clip(s.x + K @ (y - s.C @ s.x), 0, 0.75)
        s.P = (np.eye(3) - K @ s.C) @ s.P; s.P = 0.5*(s.P + s.P.T)
        return s.x.copy()

def replay(d, Q, R):
    h1 = outlier_filter(d['h1']); h3 = outlier_filter(d['h3'])
    n = len(d['T'])
    ekf = EKF([h1[0], d['ref'][0], h3[0]], np.diag([(5e-3)**2,(20e-3)**2,(5e-3)**2]), Q, R)
    X = np.zeros((n,3)); X[0] = ekf.x; uprev = np.zeros(2)
    for k in range(n-1):
        ekf.predict(uprev); X[k+1] = ekf.update(np.array([h1[k+1], h3[k+1]])); uprev = d['u'][k].copy()
    return X[:,1]

def run_num(p): m = re.search(r'run(\d+)', os.path.basename(p)); return int(m.group(1)) if m else -1

# AS-RUN gains
Q_asrun = np.diag([(0.5e-3)**2, (2.0e-3)**2, (0.5e-3)**2])
R_asrun = np.diag([(1.5e-3)**2, (1.5e-3)**2])

rmse = []
for p in sorted(glob.glob(os.path.join(CSV_DIR, 'ekf_run*.csv')), key=run_num):
    df = pd.read_csv(p); df.columns = df.columns.str.strip()
    d = dict(T=df['t_s'].values, h1=df['h1_meas_m'].values, h3=df['h3_meas_m'].values,
             h2r=df['h2_real_m'].values, ref=df['ref_m'].values,
             u=np.column_stack([df['u1'].values, df['u3'].values]))
    h2_est = replay(d, Q_asrun, R_asrun)
    m = mask_paper(d['T']); e = (h2_est - d['h2r'])*100
    rmse.append(float(np.sqrt(np.mean(e[m]**2))))
    print(f"{os.path.basename(p):<24} RMSE_obs = {rmse[-1]:.3f} cm")

print(f"\nEKF as-run replay RMSE_obs = {np.mean(rmse):.3f} ± {np.std(rmse, ddof=1):.3f} cm")
print(f"(vs. live EKF: 1.412 ± 0.064 cm)")
print(f"(vs. live Koopman: 1.139 ± 0.057 cm; Koopman replay: 1.151 ± 0.048 cm)")

ekf_run1.csv             RMSE_obs = 1.339 cm
ekf_run2.csv             RMSE_obs = 1.433 cm
ekf_run3.csv             RMSE_obs = 1.507 cm
ekf_run4.csv             RMSE_obs = 1.370 cm
ekf_run5.csv             RMSE_obs = 1.410 cm

EKF as-run replay RMSE_obs = 1.412 ± 0.064 cm
(vs. live EKF: 1.412 ± 0.064 cm)
(vs. live Koopman: 1.139 ± 0.057 cm; Koopman replay: 1.151 ± 0.048 cm)


In [3]:
"""
decay_check.py — Verifica el tiempo al 10% del offset +2cm en h2_hat(0)
y rellena los [[XX]] del paper.

Uso (Jupyter o consola):
    %run decay_check.py
    o
    python decay_check.py "C:\\Users\\PC\\Downloads\\articulo"

Reporta:
  1. time_to_10pct con telemetría real (baseline vs inyección) por corrida
  2. time_to_10pct con propagación libre e_k = A_c^k e_0
  3. ρ(A_c) y τ_e
  4. Descomposición modal del offset inicial
"""
import sys, os, glob, re
import numpy as np
import pandas as pd

# ============================================================================
# Matrices del Apéndice A
# ============================================================================
A = np.array([
 [ 0.7941, 0.1507, 0.0724, 0.0216, 0.0302, 0.0344, 0.0143,-0.0942],
 [ 0.1946, 0.7226, 0.0680, 0.0548, 0.0739,-0.1497, 0.0084, 0.0127],
 [ 0.0365, 0.0833, 0.8535, 0.0173,-0.0707,-0.0142,-0.0340, 0.0932],
 [-0.0516, 0.0113, 0.0399, 0.5721, 0.3169, 0.1792, 0.1078,-0.1236],
 [ 0.0516, 0.0042,-0.0619, 0.3049, 0.3937, 0.2134,-0.1147, 0.1401],
 [ 0.0352,-0.0175,-0.0259, 0.1833, 0.1875, 0.5980,-0.0418, 0.0639],
 [-0.0316, 0.0987,-0.0783, 0.2436,-0.0136,-0.2643, 0.8455, 0.1724],
 [-0.0984, 0.0471, 0.0607,-0.0872, 0.1991,-0.0421, 0.1493, 0.8004]])
B = np.array([
 [ 0.0194,-0.0005],[ 0.0023, 0.0007],[ 0.0001, 0.0211],[ 0.0029, 0.0002],
 [ 0.0003, 0.0026],[ 0.0022, 0.0028],[ 0.0159,-0.0001],[ 0.0001, 0.0152]])
L = np.array([
 [ 0.5225, 0.0485, 0.0257,-0.0442],[ 0.2277, 0.1175, 0.1043, 0.0318],
 [ 0.0295, 0.5365,-0.0048, 0.0543],[ 0.1977,-0.1857, 0.0683, 0.0538],
 [ 0.2178,-0.2120,-0.0837, 0.1917],[ 0.2155,-0.2040,-0.0759, 0.1508],
 [-0.0023,-0.0319, 0.5867, 0.1208],[-0.0441, 0.0348, 0.1149, 0.4999]])
C = np.zeros((4, 8))
for i, j in enumerate([0, 2, 6, 7]):
    C[i, j] = 1.0
TS = 2.0
EPS_PSI = 1e-8
A_c = (np.eye(8) - L @ C) @ A

def psi(h1, h2, h3):
    return np.array([np.sqrt(max(h1, EPS_PSI)), np.sqrt(max(h2, EPS_PSI)),
                     np.sqrt(max(h3, EPS_PSI)),
                     h1*h2, h2*h3, h1*h3, h1, h3])

def run_replay(y, u, z0):
    """Replay del observador con telemetría real (y_k, u_k)."""
    n = len(y)
    z = z0.copy()
    out = np.zeros(n)
    uprev = np.zeros(2)
    for k in range(n):
        zp = A @ z + B @ uprev
        zp[:3] = np.maximum(zp[:3], 0)
        zp[6:] = np.maximum(zp[6:], 0)
        z = zp + L @ (y[k] - C @ zp)
        z[:3] = np.maximum(z[:3], 0)
        z[6:] = np.maximum(z[6:], 0)
        out[k] = z[1]**2
        uprev = u[k].copy()
    return out

def time_to_10pct(delta, offset=0.02, frac=0.10):
    """Primer k tal que |Δ| <= frac*offset para todo t >= k."""
    thr = frac * offset
    ok = np.abs(delta) <= thr
    for k in range(len(ok)):
        if ok[k:].all():
            return k * TS
    return float('nan')

def load_log(path):
    df = pd.read_csv(path); df.columns = df.columns.str.strip()
    return dict(
        T=df['t_s'].values,
        h1=df['h1_meas_m'].values, h3=df['h3_meas_m'].values,
        h2r=df['h2_real_m'].values, ref=df['ref_m'].values,
        u=np.column_stack([df['u1'].values, df['u3'].values]))

def build_y(d):
    """y_k = [√h1, √h3, h1, h3] (4xN)."""
    n = len(d['h1'])
    y = np.zeros((n, 4))
    y[:, 0] = np.sqrt(np.maximum(d['h1'], EPS_PSI))
    y[:, 1] = np.sqrt(np.maximum(d['h3'], EPS_PSI))
    y[:, 2] = d['h1']
    y[:, 3] = d['h3']
    return y

def run_num(p):
    m = re.search(r'run(\d+)', os.path.basename(p))
    return int(m.group(1)) if m else -1

# ============================================================================
# MAIN
# ============================================================================
if __name__ == "__main__":
    # Jupyter pasa -f=kernel.json; filtrar
    argv = [a for a in sys.argv[1:] if not a.startswith('-')]
    CSV_DIR = argv[0] if argv else r"C:\Users\PC\Downloads\articulo"

    print("=" * 78)
    print(f" decay_check.py  (dir: {CSV_DIR})")
    print("=" * 78)

    # === 1. Propiedades espectrales ==========================================
    eigs = np.abs(np.linalg.eigvals(A_c))
    rho = eigs.max()
    tau_e = -TS / np.log(rho)
    print(f"\nρ(A_c) = {rho:.6f}   τ_e = {tau_e:.2f} s")
    print(f"|λ(A_c)| ordenados: {np.sort(eigs)[::-1]}")
    print(f"modos (τ en s): {np.sort(-TS/np.log(eigs[eigs>0]))[::-1]}")

    # === 2. Descomposición modal del offset +2cm en h2 =======================
    print("\n" + "-" * 78)
    print(" Descomposición modal del offset +2cm en h2(0)")
    print("-" * 78)
    for h2_op in [0.10, 0.15, 0.20]:
        h_op = np.array([h2_op * 1.3, h2_op, h2_op * 1.1])  # h1≈1.3·h2, h3≈1.1·h2
        z0_base = psi(*h_op)
        z0_inj = psi(h_op[0], h_op[1] + 0.02, h_op[2])
        e0 = z0_inj - z0_base

        # Proyección sobre autovectores
        w, V = np.linalg.eig(A_c)
        coeffs = np.linalg.solve(V, e0)
        weights = np.abs(coeffs) / np.abs(coeffs).sum()
        print(f"\n  h2_op = {h2_op:.2f} m  |Δz(0)| = {np.linalg.norm(e0):.4f}")
        idx = np.argsort(np.abs(w))[::-1]
        for i in idx[:4]:
            tau_i = -TS / np.log(np.abs(w[i])) if np.abs(w[i]) < 1 else np.inf
            print(f"    modo |λ|={np.abs(w[i]):.4f} (τ={tau_i:6.1f} s): "
                  f"{weights[i]*100:5.1f}% de la perturbación")

    # === 3. Propagación libre (determinística) ===============================
    print("\n" + "-" * 78)
    print(" Propagación libre e_k = A_c^k e_0  (sin telemetría)")
    print("-" * 78)
    for h2_op in [0.10, 0.15, 0.20]:
        h_op = np.array([h2_op * 1.3, h2_op, h2_op * 1.1])
        z0_base = psi(*h_op)
        z0_inj = psi(h_op[0], h_op[1] + 0.02, h_op[2])
        e0 = z0_inj - z0_base

        n = 400
        delta = np.zeros(n)
        e = e0.copy()
        for k in range(n):
            delta[k] = (z0_base[1] + e[1])**2 - z0_base[1]**2
            e = A_c @ e
        t10 = time_to_10pct(delta)
        print(f"  h2_op = {h2_op:.2f} m  → t(10%) = {t10:.0f} s")

    # === 4. Replay con telemetría real =======================================
    print("\n" + "-" * 78)
    print(" Replay con telemetría real (Koopman + EKF)")
    print("-" * 78)

    logs = sorted(glob.glob(os.path.join(CSV_DIR, '*.csv')), key=run_num)
    if not logs:
        print(f"  ⚠ No hay CSVs en {CSV_DIR}")
        sys.exit(0)

    results = {'koopman': [], 'ekf': []}
    for path in logs:
        name = os.path.basename(path).lower()
        method = 'koopman' if 'koopman' in name else ('ekf' if 'ekf' in name else None)
        if method is None:
            continue
        try:
            d = load_log(path)
            y = build_y(d)
            # Baseline: h2(0) = ref(0) (config experimental)
            z0_base = psi(d['h1'][0], d['ref'][0], d['h3'][0])
            # Inyección: h2(0) = ref(0) + 2cm
            z0_inj = psi(d['h1'][0], d['ref'][0] + 0.02, d['h3'][0])

            h2_base = run_replay(y, d['u'], z0_base)
            h2_inj = run_replay(y, d['u'], z0_inj)
            delta = h2_inj - h2_base
            t10 = time_to_10pct(delta)
            results[method].append((run_num(path), t10))
            print(f"  {os.path.basename(path):<24} t(10%) = {t10:6.0f} s")
        except Exception as e:
            print(f"  {os.path.basename(path):<24} error: {type(e).__name__}: {e}")

    print("\n" + "=" * 78)
    print(" RESUMEN")
    print("=" * 78)
    for method, vals in results.items():
        if not vals:
            continue
        arr = np.array([v[1] for v in vals if np.isfinite(v[1])])
        if len(arr):
            print(f"  {method:<8}  t(10%) = {arr.mean():6.0f} ± {arr.std():4.0f} s "
                  f"(min {arr.min():.0f}, max {arr.max():.0f}, n={len(arr)})")

    print(f"\n  τ_e (asintótico de A_c)    = {tau_e:.2f} s")
    print(f"  2.3·τ_e (10% de respuesta pura) = {2.3*tau_e:.1f} s")
    print(f"  ρ(A_c)                     = {rho:.5f}")
    print()
    print("  → Rellena en el paper:")
    print("    [[XX]] (propagación libre)  = ver arriba")
    print("    [[CONFIRM replay 78s]]      = ver arriba")
    print("    [[XX]] (Koopman replay own) = ver arriba")
    print("    [[XX]] (EKF replay all)     = ver arriba")

 decay_check.py  (dir: C:\Users\PC\Downloads\articulo)

ρ(A_c) = 0.935146   τ_e = 29.83 s
|λ(A_c)| ordenados: [0.93514625 0.65652243 0.48154553 0.4092778  0.38674089 0.34959571
 0.34959571 0.03428874]
modos (τ en s): [29.82744318  4.75286953  2.73689728  2.23873629  2.1052624   1.90298957
  1.90298957  0.59295481]

------------------------------------------------------------------------------
 Descomposición modal del offset +2cm en h2(0)
------------------------------------------------------------------------------

  h2_op = 0.10 m  |Δz(0)| = 0.0304
    modo |λ|=0.9351 (τ=  29.8 s):   5.2% de la perturbación
    modo |λ|=0.6565 (τ=   4.8 s):  57.1% de la perturbación
    modo |λ|=0.4815 (τ=   2.7 s):   3.6% de la perturbación
    modo |λ|=0.4093 (τ=   2.2 s):   2.2% de la perturbación

  h2_op = 0.15 m  |Δz(0)| = 0.0255
    modo |λ|=0.9351 (τ=  29.8 s):   1.4% de la perturbación
    modo |λ|=0.6565 (τ=   4.8 s):  56.5% de la perturbación
    modo |λ|=0.4815 (τ=   2.7 s):   5.2% de la

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CSV_DIR = Path(r"C:\Users\PC\Downloads\articulo")
IMG_DIR = Path("img")  # ajusta si tus PNGs están en otra carpeta
IMG_DIR.mkdir(exist_ok=True)


def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df


def unified_mask(T, settling=60.0):
    """Ventana unificada: excluir [0,60), [264,324), [528,588)."""
    m = np.ones(len(T), dtype=bool)
    for tc in (0.0, 264.0, 528.0):
        m &= ~((T >= tc) & (T < tc + settling))
    return m


def make_figure(csv_path, method_label, est_col, out_png):
    df = load_csv(csv_path)
    T = df['t_s'].values
    h1 = df['h1_meas_m'].values * 100
    h2_est = df[est_col].values * 100
    h2_real = df['h2_real_m'].values * 100
    h3 = df['h3_meas_m'].values * 100
    ref = df['ref_m'].values * 100
    u1 = df['u1'].values
    u3 = df['u3'].values

    mask = unified_mask(T)
    rmse_obs = np.sqrt(np.mean(((h2_est - h2_real)[mask])**2))
    rmse_trk = np.sqrt(np.mean(((h2_est - ref)[mask])**2))

    fig, ax = plt.subplots(3, 2, figsize=(14, 11))
    fig.suptitle(
        f'{method_label} | RMSE obs={rmse_obs:.3f} cm | RMSE track={rmse_trk:.3f} cm',
        fontsize=10, fontweight='bold')

    # Panel 1: h1
    ax[0, 0].plot(T, h1, 'b', lw=1.5, label='$h_1$ measured')
    ax[0, 0].set_ylabel('$h_1$ [cm]'); ax[0, 0].set_xlabel('t [s]')
    ax[0, 0].legend(fontsize=8); ax[0, 0].grid(alpha=0.3)

    # Panel 2: h2 estimated vs ground truth vs ref
    ax[0, 1].plot(T, h2_est, 'r--', lw=2, label=f'$h_2$ estimated ({method_label})')
    ax[0, 1].plot(T, h2_real, 'g-', lw=1.2, alpha=0.85, label='$h_2$ ground truth')
    ax[0, 1].step(T, ref, where='post', color='k', ls='--', lw=1.5, label='Reference')  # <-- fix
    ax[0, 1].set_ylabel('$h_2$ [cm]'); ax[0, 1].set_xlabel('t [s]')
    ax[0, 1].legend(fontsize=8); ax[0, 1].grid(alpha=0.3)

    # Panel 3: h3
    ax[1, 0].plot(T, h3, 'b', lw=1.5, label='$h_3$ measured')
    ax[1, 0].set_ylabel('$h_3$ [cm]'); ax[1, 0].set_xlabel('t [s]')
    ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=0.3)

    # Panel 4: tracking error con bandas
    e_trk = h2_est - ref
    ax[1, 1].plot(T, e_trk, 'r', lw=1.5, label=f'RMSE_SS={rmse_trk:.3f} cm')
    ax[1, 1].axhline(0, color='k', ls=':')
    for tc in (0.0, 264.0, 528.0):
        ax[1, 1].axvspan(tc, tc + 60.0, alpha=0.12, color='orange')
    ax[1, 1].set_ylabel('Ctrl error $h_2$ [cm]'); ax[1, 1].set_xlabel('t [s]')
    ax[1, 1].legend(fontsize=8); ax[1, 1].grid(alpha=0.3)

    # Panel 5: control
    ax[2, 0].step(T, u1, where='post', color='b', lw=1.5, label='$u_1$')  # <-- fix
    ax[2, 0].step(T, u3, where='post', color='r', lw=1.5, label='$u_3$')  # <-- fix
    ax[2, 0].set_ylim(-0.05, 1.1)
    ax[2, 0].set_ylabel('Control [0–1]'); ax[2, 0].set_xlabel('t [s]')
    ax[2, 0].legend(fontsize=8); ax[2, 0].grid(alpha=0.3)

    # Panel 6: "Step interval" (NO "Computation time")
    ax[2, 1].text(
        0.5, 0.5,
        'Step interval ≈ 2100 ms\n(ESP32 serial-limited, not algorithmic)',
        ha='center', va='center', fontsize=12,
        transform=ax[2, 1].transAxes)
    ax[2, 1].set_title('Step interval (ESP32-limited)')
    ax[2, 1].axis('off')

    plt.tight_layout()
    out_path = IMG_DIR / out_png
    plt.savefig(out_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"✓ {out_path}  | RMSE obs={rmse_obs:.3f} cm")


# Generar las tres figuras
make_figure(
    CSV_DIR / "koopman_run4.csv",
    "Koopman run 4", "h2_est_m",
    "koopman_best_run.png")

make_figure(
    CSV_DIR / "ekf_run5.csv",
    "EKF run 5", "h2_ekf_m",       # <- ajusta si la columna se llama distinto
    "ekf_best_run.png")

# Para la figura overlay (Fig 6), es una superposición: dos runs en el mismo plot
# Haremos un plot específico solo con los paneles de h2 y ref de los dos runs
def make_overlay():
    df_k = load_csv(CSV_DIR / "koopman_run4.csv")
    df_e = load_csv(CSV_DIR / "ekf_run5.csv")
    T = df_k['t_s'].values
    h2_k = df_k['h2_est_m'].values * 100
    h2_e = df_e['h2_ekf_m'].values * 100
    h2_real_k = df_k['h2_real_m'].values * 100   # GT de Koopman run 4
    h2_real_e = df_e['h2_real_m'].values * 100   # GT de EKF run 5 (su propia GT)
    ref = df_k['ref_m'].values * 100

    mask = unified_mask(T)
    # Cada estimación se compara contra SU PROPIA ground truth
    rmse_k = np.sqrt(np.mean(((h2_k - h2_real_k)[mask])**2))  # 1.109 cm
    rmse_e = np.sqrt(np.mean(((h2_e - h2_real_e)[mask])**2))  # 1.410 cm

    fig, ax = plt.subplots(figsize=(10, 4.5))
    # Ground truth de Koopman run 4 (sólida) y de EKF run 5 (dashed, para comparar)
    ax.plot(T, h2_real_k, 'g-',  lw=1.5, label='Ground truth (Koopman run 4)')
    ax.plot(T, h2_real_e, 'g--', lw=1.2, alpha=0.6, label='Ground truth (EKF run 5)')
    ax.plot(T, h2_k, 'b--', lw=1.8, label=f'Koopman run 4 (obs RMSE={rmse_k:.3f} cm)')
    ax.plot(T, h2_e, 'r:',  lw=1.8, label=f'EKF run 5 (obs RMSE={rmse_e:.3f} cm)')
    ax.step(T, ref, where='post', color='k', ls='--', lw=1.2, alpha=0.6, label='Reference')
    ax.set_xlabel('Time [s]'); ax.set_ylabel('$h_2$ [cm]')
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMG_DIR / "overlay_best_runs.png", dpi=200, bbox_inches='tight')
    plt.close()
    print(f"✓ Overlay regenerado | Koopman run 4: {rmse_k:.3f} cm | EKF run 5: {rmse_e:.3f} cm")

make_overlay()

✓ img\koopman_best_run.png  | RMSE obs=1.109 cm
✓ img\ekf_best_run.png  | RMSE obs=1.410 cm
✓ Overlay regenerado | Koopman run 4: 1.109 cm | EKF run 5: 1.410 cm


In [2]:
"""
validate_residual.py — Verifica el residuo algebraico de los observables elevados
==============================================================================

Calcula r4 = z4 - z1²·z2² (y análogos r5, r6) sobre los logs Koopman en la
ventana unificada de 310 muestras. Reporta RMS absoluto [m²] y RMS relativo
respecto a RMS(z4).

Uso:
    python validate_residual.py                    # usa la carpeta por defecto
    python validate_residual.py "C:/ruta/articulo"

Salida: tabla por run + interpretación + snippet LaTeX listo para pegar.
"""
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# Matrices del Apéndice A
# ============================================================================
A = np.array([
    [ 0.7941,  0.1507,  0.0724,  0.0216,  0.0302,  0.0344,  0.0143, -0.0942],
    [ 0.1946,  0.7226,  0.0680,  0.0548,  0.0739, -0.1497,  0.0084,  0.0127],
    [ 0.0365,  0.0833,  0.8535,  0.0173, -0.0707, -0.0142, -0.0340,  0.0932],
    [-0.0516,  0.0113,  0.0399,  0.5721,  0.3169,  0.1792,  0.1078, -0.1236],
    [ 0.0516,  0.0042, -0.0619,  0.3049,  0.3937,  0.2134, -0.1147,  0.1401],
    [ 0.0352, -0.0175, -0.0259,  0.1833,  0.1875,  0.5980, -0.0418,  0.0639],
    [-0.0316,  0.0987, -0.0783,  0.2436, -0.0136, -0.2643,  0.8455,  0.1724],
    [-0.0984,  0.0471,  0.0607, -0.0872,  0.1991, -0.0421,  0.1493,  0.8004]])
B = np.array([
    [ 0.0194, -0.0005],[ 0.0023,  0.0007],[ 0.0001,  0.0211],[ 0.0029,  0.0002],
    [ 0.0003,  0.0026],[ 0.0022,  0.0028],[ 0.0159, -0.0001],[ 0.0001,  0.0152]])
L = np.array([
    [ 0.5225,  0.0485,  0.0257, -0.0442],[ 0.2277,  0.1175,  0.1043,  0.0318],
    [ 0.0295,  0.5365, -0.0048,  0.0543],[ 0.1977, -0.1857,  0.0683,  0.0538],
    [ 0.2178, -0.2120, -0.0837,  0.1917],[ 0.2155, -0.2040, -0.0759,  0.1508],
    [-0.0023, -0.0319,  0.5867,  0.1208],[-0.0441,  0.0348,  0.1149,  0.4999]])
C = np.zeros((4, 8))
for i, j in enumerate([0, 2, 6, 7]):
    C[i, j] = 1.0

EPS_PSI = 1e-8


# ============================================================================
# Utilidades
# ============================================================================
def psi(h1, h2, h3):
    """Vector de 8 observables del paper."""
    return np.array([
        np.sqrt(max(h1, EPS_PSI)),
        np.sqrt(max(h2, EPS_PSI)),
        np.sqrt(max(h3, EPS_PSI)),
        h1*h2, h2*h3, h1*h3,
        h1, h3
    ])


def replay_koopman(d):
    """Replay del observador Koopman sobre un log. Devuelve Z de forma (n, 8)."""
    h1, h3, ref, u = d['h1'], d['h3'], d['ref'], d['u']
    n = len(h1)
    z = psi(h1[0], ref[0], h3[0])
    Z = np.zeros((n, 8))
    Z[0] = z
    uprev = np.zeros(2)
    for k in range(n - 1):
        zp = A @ z + B @ uprev
        zp[:3] = np.maximum(zp[:3], 0)
        zp[6:] = np.maximum(zp[6:], 0)
        y = np.array([
            np.sqrt(max(h1[k+1], EPS_PSI)),
            np.sqrt(max(h3[k+1], EPS_PSI)),
            h1[k+1], h3[k+1]
        ])
        z = zp + L @ (y - C @ zp)
        z[:3] = np.maximum(z[:3], 0)
        z[6:] = np.maximum(z[6:], 0)
        Z[k+1] = z
        uprev = u[k].copy()
    return Z


def mask_ss(T, settling=60.0):
    """Ventana unificada de 310 muestras (excluye 60 s tras cada referencia)."""
    m = np.ones(len(T), dtype=bool)
    for tc in (0.0, 264.0, 528.0):
        m &= ~((T >= tc) & (T < tc + settling))
    return m


# ============================================================================
# Main
# ============================================================================
def main():
    argv = [a for a in sys.argv[1:] if not a.startswith('-')]
    csv_dir = Path(argv[0]) if argv else Path(r"C:\Users\PC\Downloads\articulo")

    if not csv_dir.is_dir():
        print(f"✗ Directorio no encontrado: {csv_dir}")
        return

    logs = sorted(csv_dir.glob("koopman_run*.csv"))
    if not logs:
        print(f"✗ No hay koopman_run*.csv en {csv_dir}")
        return

    print("=" * 95)
    print(f" Verificación de residuo algebraico  (dir: {csv_dir})")
    print(" r4 = z4 − z1²z2²     r5 = z5 − z2²z3²     r6 = z6 − z1²z3²")
    print("=" * 95)
    print(f"\n{'run':<22} {'RMS(r4) [m²]':>14} {'RMS(z4) [m²]':>14} {'rel r4 [%]':>11}"
          f" {'rel r5 [%]':>11} {'rel r6 [%]':>11}")
    print("-" * 95)

    all_r4_abs, all_z4_abs = [], []
    all_r4_rel, all_r5_rel, all_r6_rel = [], [], []

    for log in logs:
        df = pd.read_csv(log)
        df.columns = df.columns.str.strip()
        d = dict(
            T=df['t_s'].values,
            h1=df['h1_meas_m'].values,
            h3=df['h3_meas_m'].values,
            ref=df['ref_m'].values,
            u=np.column_stack([df['u1'].values, df['u3'].values]),
        )
        Z = replay_koopman(d)
        m = mask_ss(d['T'])

        # Residuos algebraicos (todas las variables en m²)
        r4 = Z[:, 3] - Z[:, 0]**2 * Z[:, 1]**2   # z4 = h1·h2 vs (√h1)²·(√h2)²
        r5 = Z[:, 4] - Z[:, 1]**2 * Z[:, 2]**2   # z5 = h2·h3 vs (√h2)²·(√h3)²
        r6 = Z[:, 5] - Z[:, 0]**2 * Z[:, 2]**2   # z6 = h1·h3 vs (√h1)²·(√h3)²
        z4, z5, z6 = Z[:, 3], Z[:, 4], Z[:, 5]

        # Estadísticos en la ventana de estado estacionario
        rms_r4 = np.sqrt(np.mean(r4[m]**2))
        rms_r5 = np.sqrt(np.mean(r5[m]**2))
        rms_r6 = np.sqrt(np.mean(r6[m]**2))
        rms_z4 = np.sqrt(np.mean(z4[m]**2))
        rms_z5 = np.sqrt(np.mean(z5[m]**2))
        rms_z6 = np.sqrt(np.mean(z6[m]**2))

        rel_r4 = 100.0 * rms_r4 / rms_z4 if rms_z4 > 0 else np.nan
        rel_r5 = 100.0 * rms_r5 / rms_z5 if rms_z5 > 0 else np.nan
        rel_r6 = 100.0 * rms_r6 / rms_z6 if rms_z6 > 0 else np.nan

        all_r4_abs.append(rms_r4)
        all_z4_abs.append(rms_z4)
        all_r4_rel.append(rel_r4)
        all_r5_rel.append(rel_r5)
        all_r6_rel.append(rel_r6)

        print(f"{log.name:<22} {rms_r4:>14.4e} {rms_z4:>14.4e} {rel_r4:>11.2f}"
              f" {rel_r5:>11.2f} {rel_r6:>11.2f}")

    # Estadísticos agregados
    print("-" * 95)
    print(f"{'MEDIA':<22} {np.mean(all_r4_abs):>14.4e} {np.mean(all_z4_abs):>14.4e}"
          f" {np.mean(all_r4_rel):>11.2f} {np.mean(all_r5_rel):>11.2f} {np.mean(all_r6_rel):>11.2f}")
    print(f"{'STD (ddof=1)':<22} {np.std(all_r4_abs, ddof=1):>14.4e} {np.std(all_z4_abs, ddof=1):>14.4e}"
          f" {np.std(all_r4_rel, ddof=1):>11.2f} {np.std(all_r5_rel, ddof=1):>11.2f} {np.std(all_r6_rel, ddof=1):>11.2f}")

    # ========================================================================
    # Interpretación
    # ========================================================================
    mean_r4_abs = float(np.mean(all_r4_abs))
    mean_z4_abs = float(np.mean(all_z4_abs))
    mean_r4_rel = float(np.mean(all_r4_rel))
    ratio = mean_r4_abs / mean_z4_abs if mean_z4_abs > 0 else float('inf')

    print("\n" + "=" * 95)
    print(" Interpretación")
    print("=" * 95)
    print(f"\n  RMS(r4) absoluto  = {mean_r4_abs:.4e} m²")
    print(f"  RMS(z4) absoluto  = {mean_z4_abs:.4e} m²")
    print(f"  RMS(r4)/RMS(z4)   = {ratio:.4f}  (i.e., {mean_r4_rel:.2f}%)")

    print("\n  Escenarios:")
    if 2.8 < mean_r4_rel < 3.6 and mean_r4_abs < 1e-3:
        print(f"  ✓ CASO 1: el paper dice '3.22e-2 m²' pero el valor que se reprodujo es el RELATIVO")
        print(f"    ({mean_r4_rel:.2f}%). El valor absoluto real es {mean_r4_abs:.2e} m².")
        print(f"    → Corregir el paper: cambiar '3.22×10⁻² m²' por 'a relative RMS residual of {mean_r4_rel:.2f}%'")
    elif 2.8 < mean_r4_rel < 3.6 and mean_r4_abs > 1e-3:
        print(f"  ⚠ CASO 2: RMS relativo ≈ 3.22% PERO absoluto = {mean_r4_abs:.2e} m²")
        print(f"    La variable z4 = {mean_z4_abs:.2e} m². Revisar unidades.")
        print(f"    → Decidir si el paper reporta el valor absoluto ({mean_r4_abs:.2e} m²)")
        print(f"      o el relativo ({mean_r4_rel:.2f}%).")
    elif mean_r4_abs > 1e-3 and mean_r4_rel > 10:
        print(f"  ✗ CASO 3: el residuo es comparable a z4 (ratio {ratio:.2f}).")
        print(f"    z4 = {mean_z4_abs:.2e} m²  vs.  r4 = {mean_r4_abs:.2e} m²")
        print(f"    → El valor '3.22×10⁻² m²' del paper NO se reproduce.")
        print(f"    → Reformular como 'a relative RMS residual of {mean_r4_rel:.2f}%'.")
    else:
        print(f"  ? CASO NO ESPERADO. Reportar los valores crudos tal cual.")

    # ========================================================================
    # LaTeX listo para pegar
    # ========================================================================
    print("\n" + "=" * 95)
    print(" LaTeX sugerido para §5.2")
    print("=" * 95)

    if mean_r4_abs < 1e-3 and mean_r4_rel < 10:
        # Caso limpio: residuo absoluto pequeño, relativo razonable
        latex = (
            f"Specifically, tracking the nonlinear algebraic residual "
            f"$z_4-z_1^2z_2^2$ across the experimental telemetry reveals a\n"
            f"non-negligible relative RMS discrepancy of ${mean_r4_rel:.2f}\\%$ "
            f"(absolute RMS ${mean_r4_abs:.2e}$~m$^2$ against "
            f"$\\mathrm{{RMS}}(z_4)={mean_z4_abs:.2e}$~m$^2$). If the physical state\n"
            f"is re-lifted at every time step ($\\hat z_k\\leftarrow\\Psi(\\hat x_k)$) "
            f"to force algebraic consistency, the observation RMSE degrades by\n"
            f"$+10.3\\%$."
        )
    else:
        # Caso en que el residuo es grande: usar solo el relativo
        latex = (
            f"Specifically, tracking the nonlinear algebraic residual "
            f"$z_4-z_1^2z_2^2$ across the experimental telemetry reveals a\n"
            f"relative RMS discrepancy of ${mean_r4_rel:.2f}\\%$ with respect to "
            f"$\\mathrm{{RMS}}(z_4)$. If the physical state is re-lifted at\n"
            f"every time step ($\\hat z_k\\leftarrow\\Psi(\\hat x_k)$) to force "
            f"algebraic consistency, the observation RMSE degrades by $+10.3\\%$."
        )

    print("\n" + latex + "\n")
    print("=" * 95)


if __name__ == "__main__":
    main()

 Verificación de residuo algebraico  (dir: C:\Users\PC\Downloads\articulo)
 r4 = z4 − z1²z2²     r5 = z5 − z2²z3²     r6 = z6 − z1²z3²

run                      RMS(r4) [m²]   RMS(z4) [m²]  rel r4 [%]  rel r5 [%]  rel r6 [%]
-----------------------------------------------------------------------------------------------
koopman_run1.csv           3.1543e-02     2.5601e-02      123.21      116.60      113.39
koopman_run2.csv           3.3435e-02     2.6904e-02      124.28      119.88      116.83
koopman_run3.csv           3.3776e-02     2.6198e-02      128.93      125.00      121.63
koopman_run4.csv           3.2749e-02     2.6804e-02      122.18      116.71      113.70
koopman_run5.csv           3.3739e-02     2.6970e-02      125.10      120.65      117.69
-----------------------------------------------------------------------------------------------
MEDIA                      3.3048e-02     2.6495e-02      124.74      119.77      116.65
STD (ddof=1)               9.3715e-04     5.8650e